In [ ]:
import torch
import torch.nn as nn
from collections import deque

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

relu = nn.ReLU

cuda


In [ ]:
def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    
    pos = torch.arange(0, max_seq_len).unsqueeze(1)  # (max_seq_len, 1)
    i = torch.arange(0, d_model, 2)                  # (d_model/2,)
    
    div = 10000 ** (i / d_model)                     # (d_model/2,)
    
    PE[:, 0::2] = torch.sin(pos / div)               # even columns
    PE[:, 1::2] = torch.cos(pos / div)               # odd columns
    
    return PE

In [ ]:
class Attention_Block(nn.Module):
  def __init__(self, d_model:int, num_heads:int):
    super().__init__()
    self.num_heads = num_heads
    self.input = None
    self.Exp = None
    self.d_k = d_model // num_heads
    self.d_model = d_model
    self.Q_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.K_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.V_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.W_O = nn.Parameter(torch.randn(d_model, d_model))
    self.Q = None
    self.K = None
    self.V = None
    self.scores = None
    self.A = None
    self.deltaE_heads = None
    self.deltaE_cat = None

  def forward_pass(self, E, mask = False):
    n = E.shape[0]
    self.inputs = E
    self.Exp = self.inputs.unsqueeze(0).expand(self.num_heads, -1, -1)
    self.Q = torch.bmm(self.Exp, self.Q_M)
    self.K = torch.bmm(self.Exp, self.K_M)
    self.V = torch.bmm(self.Exp, self.V_M)

    self.scores = torch.bmm(self.Q, self.K.transpose(-2, -1)) / (self.d_k) ** 0.5
    if mask:
      m = torch.triu(torch.ones(n, n), diagonal=1).bool().to(self.Q_M.device)
      self.scores = self.scores.masked_fill(m, float('-inf')).to(device)

    self.A = torch.softmax(self.scores, dim = -1)
    self.deltaE_heads = torch.bmm(self.A, self.V)
    self.deltaE_cat = self.deltaE_heads.transpose(0, 1).reshape(n, -1)

    return self.deltaE_cat @ self.W_O

class Layer(nn.Module):
  def __init__(self, input, prev):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(prev, input))
    self.bias = nn.Parameter(torch.randn(input))
    self.activations = None
    self.z = None
    self.inputs = input
    self.prev = prev

class MLP(nn.Module):
  def __init__(self, num_layers:int, layers:list):
    super().__init__()
    self.num = num_layers
    self.layers = nn.ModuleList()
    self.l = layers
    self.input = None
    for i in range(1, self.num + 1):
      self.layers.append(Layer(layers[i], layers[i - 1]))

  def forward_pass(self, input):
    self.layers[0].activations = input
    self.input = input
    for i in range(1, self.num):
      self.layers[i].z = (self.layers[i-1].activations @ self.layers[i].weights) + self.layers[i].bias
      self.layers[i].activations = relu(self.layers[i].z)

    last = self.num
    self.layers[last].z = self.layers[last-1].activations @ self.layers[last].weights + self.layers[last].bias
    self.layers[last].activations = self.layers[last].z

    return self.layers[self.num].activations

class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gamma = nn.Parameter(torch.ones(d_model))   # scale
        self.beta = nn.Parameter(torch.zeros(d_model))   # shift
        # Intermediate values for backprop
        self.input = None
        self.mu = None
        self.var = None
        self.x_hat = None

    def forward_pass(self, E):
        self.input = E
        # Mean across d_model for each token row
        self.mu = E.sum(dim=-1, keepdim=True) / self.d_model          # (n, 1)
        # Variance across d_model for each token row
        self.var = ((E - self.mu) ** 2).sum(dim=-1, keepdim=True) / self.d_model  # (n, 1)
        # Normalize
        self.x_hat = (E - self.mu) / (self.var + 1e-5) ** 0.5        # (n, d_model)
        # Scale and shift
        return self.gamma * self.x_hat + self.beta                     # (n, d_model)


class Transformer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = Attention_Block(d_model, num_heads).to(device)
        self.mlp = MLP(3, [d_model, 4*d_model, 4*d_model, d_model]).to(device)
        self.norm1 = LayerNorm(d_model).to(device)  # before attention
        self.norm2 = LayerNorm(d_model).to(device)  # before MLP
        # Intermediate values for backprop
        self.input = None
        self.norm1_out = None
        self.attn_out = None
        self.norm2_out = None
        self.mlp_out = None

    def forward_pass(self, E, mask=False):
        self.input = E

        # Pre-LN attention sub-block
        self.norm1_out = self.norm1.forward_pass(E)
        self.attn_out = self.attention.forward_pass(self.norm1_out, mask)
        E = E + self.attn_out

        # Pre-LN MLP sub-block
        self.norm2_out = self.norm2.forward_pass(E)
        self.mlp_out = self.mlp.forward_pass(self.norm2_out)
        E = E + self.mlp_out

        return E

class LLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, max_seq_len):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, d_model))
        self.pos_encoding = positional_encoding(max_seq_len, d_model).to(device)
        self.blocks = nn.ModuleList([
            Transformer(d_model, num_heads)
            for _ in range(num_blocks)
        ])
        self.unembedding = nn.Parameter(torch.randn(d_model, vocab_size))
        # Store for backprop
        self.token_ids = None
        self.E = None
        self.logits = None
        self.probs = None

    def forward_pass(self, token_ids, mask=False):
        self.token_ids = token_ids
        # Embedding lookup + positional encoding
        self.E = self.embedding[token_ids] + self.pos_encoding[:len(token_ids)]
        # Transformer blocks
        for block in self.blocks:
            self.E = block.forward_pass(self.E, mask)
        # Unembedding — take last row
        self.logits = self.E[-1] @ self.unembedding       # (vocab_size,)
        self.probs = torch.softmax(self.logits, dim=-1)   # (vocab_size,)
        return self.probs